In [14]:
from common import *

In [15]:
def optimize(trial: optuna.Trial, x, y):
    criterion       = trial.suggest_categorical ("criterion",       ["gini", "entropy"])
    n_estimators    = trial.suggest_int         ("n_estimators",    100,    1500, step=100)
    max_depth       = trial.suggest_int         ("max_depth",       3,      15)
    max_features    = trial.suggest_float       ("max_features",    0.01,   1.0)
    
    model = ensemble.RandomForestClassifier(
        criterion=criterion,
        n_estimators=n_estimators,
        max_depth=max_depth,
        max_features=max_features,
    )
    kf = model_selection.StratifiedKFold(n_splits=5)
    
    accuracies = []
    for idx in kf.split(x, y):
        train_idx,  val_idx         = idx[0],       idx[1]
        train_x,    train_y         = x[train_idx], y[train_idx]
        val_x,      val_y           = x[val_idx],   y[val_idx]
        
        model.fit(train_x, train_y)
        val_preds = model.predict(val_x)
        fold_acc = metrics.accuracy_score(val_y, val_preds)
        accuracies.append(fold_acc)
    
    # -1 because we minimize this
    return -1.0 * np.mean(accuracies)

In [16]:
optimization_func = partial(
    optimize,
    x=X,
    y=y,
)

In [17]:
study = optuna.study.create_study(direction="minimize")
study.optimize(optimization_func, n_trials=15)

[I 2026-03-26 00:19:17,476] A new study created in memory with name: no-name-c16c225e-94d9-4d17-95f7-442c9551eee5
[I 2026-03-26 00:19:40,923] Trial 0 finished with value: -0.893 and parameters: {'criterion': 'gini', 'n_estimators': 1100, 'max_depth': 7, 'max_features': 0.6885983890168446}. Best is trial 0 with value: -0.893.
[I 2026-03-26 00:19:51,847] Trial 1 finished with value: -0.8915 and parameters: {'criterion': 'entropy', 'n_estimators': 600, 'max_depth': 10, 'max_features': 0.2774531247940467}. Best is trial 0 with value: -0.893.
[I 2026-03-26 00:20:02,726] Trial 2 finished with value: -0.845 and parameters: {'criterion': 'gini', 'n_estimators': 1200, 'max_depth': 11, 'max_features': 0.12730183523121044}. Best is trial 0 with value: -0.893.
[I 2026-03-26 00:20:17,964] Trial 3 finished with value: -0.8870000000000001 and parameters: {'criterion': 'gini', 'n_estimators': 1100, 'max_depth': 14, 'max_features': 0.2841221891052546}. Best is trial 0 with value: -0.893.
[I 2026-03-26 

In [18]:
study.best_params

{'criterion': 'entropy',
 'n_estimators': 600,
 'max_depth': 15,
 'max_features': 0.6755498100488898}